# Pipeline 6 - Resident Risk and Reintegration Readiness Predictor

Generated: 2026-04-07T11:53:52.838963Z


## 1) Problem Framing

**Business question:** Flag residents at elevated incident risk and estimate reintegration readiness for staff triage.

**Who cares:** nonprofit leadership, program staff, and/or fundraising staff depending on the pipeline domain.

**Why it matters:** this model turns operational, donor, or outreach data into a decision-support signal for a resource-constrained safehouse nonprofit.

**Predictive vs. explanatory goal:** this notebook includes both. The predictive model is evaluated on unseen data and is used for deployment-oriented scoring. The explanatory or relationship model is included to identify which variables appear most connected to the target and to support business interpretation. We do not treat predictive accuracy as causal proof.

**Success metrics:** classification pipelines use accuracy, F1, and ROC AUC where appropriate. Regression/forecasting pipelines use MAE, RMSE, and R-squared. The notebook also compares against a simple baseline so the results can be interpreted honestly.


## Notebook Setup

Shared imports and helper functions are defined once here so the later rubric sections can focus on the pipeline-specific code for this business problem.


In [1]:
import warnings

warnings.filterwarnings("ignore")

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Ensure the shared pipeline modules are importable when running from anywhere.
# (Repo-relative, so it works after pushing to GitHub.)
cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "ml-pipelines" else cwd
if (repo_root / "ml-pipelines").exists():
    ml_pipelines_dir = (repo_root / "ml-pipelines").resolve()
    if str(ml_pipelines_dir) not in sys.path:
        sys.path.insert(0, str(ml_pipelines_dir))

from data_prep import (
    as_bool,
    compare_profiles,
    dataset_profile,
    export_predictions_json,
    fill_numeric_median,
    get_project_paths,
    load_df,
    prep,
    quick_eda,
    safe_classifier_split,
    score_bands,
)
from modeling import (
    CandidateResult,
    ablate_feature_groups_one_by_one_cv,
    cv_evaluate_candidate,
    cv_results_table,
    evaluate_candidate,
    read_json_if_exists,
    select_simplest_within_delta,
    select_simplest_within_delta_cv,
    should_export_by_guardrail,
    top_features,
    tune_model_randomized,
    write_ablation_report_json,
    write_run_report_json,
)

paths = get_project_paths()
print("Data dir:", paths.data_dir)
print("Output dir:", paths.out_dir)


def print_business_takeaway(text):
    print("\nBusiness takeaway:")
    print(text)


Data dir: /Users/xenorex/Downloads/lighthouse_csv_v7
Output dir: /Users/xenorex/RiderProjects/INTEX2/output/ml-predictions


## 2) Data Acquisition, Preparation, and Exploration

The pipeline reads the provided CSV files from `data/raw/`, performs joins and feature engineering in code, handles missing values reproducibly, and prints an EDA summary before modeling. The EDA step checks row counts, missingness, target distributions, feature summaries, and key categorical distributions.


In [2]:
residents = load_df("residents")
incidents = load_df("incident_reports")
visits = load_df("home_visitations")
recordings = load_df("process_recordings")
edu = load_df("education_records")
health = load_df("health_wellbeing_records")

incidents["incident_date"] = pd.to_datetime(incidents["incident_date"], errors="coerce")
visits["visit_date"] = pd.to_datetime(visits["visit_date"], errors="coerce")
recordings["session_date"] = pd.to_datetime(recordings["session_date"], errors="coerce")
edu["record_date"] = pd.to_datetime(edu["record_date"], errors="coerce")
health["record_date"] = pd.to_datetime(health["record_date"], errors="coerce")
# The raw dataset has too few incidents in a 30-day holdout window to train a
# stable classifier. Use a 180-day future window as an incident-risk proxy and
# export to the app's resident_incident_30d slot with the horizon noted in payload.
incident_horizon_days = 180
cutoff = incidents["incident_date"].max() - pd.Timedelta(days=incident_horizon_days)
label_end = cutoff + pd.Timedelta(days=incident_horizon_days)
future_inc = incidents[(incidents["incident_date"] > cutoff) & (incidents["incident_date"] <= label_end)]
y = (future_inc.groupby("resident_id")["incident_id"].count() > 0).astype(int).rename("incident_next_30d").reset_index()
past_inc = incidents[incidents["incident_date"] <= cutoff].copy()
past_vis = visits[visits["visit_date"] <= cutoff].copy()
past_rec = recordings[recordings["session_date"] <= cutoff].copy()
past_edu = edu[edu["record_date"] <= cutoff].copy()
past_health = health[health["record_date"] <= cutoff].copy()

base_cols = ["resident_id","safehouse_id","case_status","case_category","is_pwd","has_special_needs","family_is_4ps","family_solo_parent","family_indigenous","family_parent_pwd","family_informal_settler","referral_source","reintegration_status","initial_risk_level","current_risk_level"]
base = residents[base_cols].copy()
for col in ["is_pwd","has_special_needs","family_is_4ps","family_solo_parent","family_indigenous","family_parent_pwd","family_informal_settler"]:
    base[col] = as_bool(base[col]).astype(int)
inc_90 = past_inc[past_inc["incident_date"] >= cutoff - pd.Timedelta(days=90)]
inc_feat = inc_90.groupby("resident_id").agg(
    incidents_90d=("incident_id","count"),
    high_severity_90d=("severity", lambda s: int((s == "High").sum())),
    unresolved_90d=("resolved", lambda s: int((~as_bool(s)).sum())),
).reset_index()
vis_90 = past_vis[past_vis["visit_date"] >= cutoff - pd.Timedelta(days=90)]
vis_feat = vis_90.groupby("resident_id").agg(
    visits_90d=("visitation_id","count"),
    safety_concerns_90d=("safety_concerns_noted", lambda s: int(as_bool(s).sum())),
    followups_90d=("follow_up_needed", lambda s: int(as_bool(s).sum())),
).reset_index()
rec_30 = past_rec[past_rec["session_date"] >= cutoff - pd.Timedelta(days=30)]
rec_feat = rec_30.groupby("resident_id").agg(
    recordings_30d=("recording_id","count"),
    progress_noted_30d=("progress_noted", lambda s: int(as_bool(s).sum())),
    concerns_flagged_30d=("concerns_flagged", lambda s: int(as_bool(s).sum())),
    referrals_30d=("referral_made", lambda s: int(as_bool(s).sum())),
).reset_index()
def latest_by_res(frame, date_col, value_cols):
    frame = frame.dropna(subset=[date_col]).sort_values(["resident_id", date_col]).copy()
    return frame.groupby("resident_id").tail(1)[["resident_id"] + value_cols] if not frame.empty else pd.DataFrame(columns=["resident_id"] + value_cols)
edu_last = latest_by_res(past_edu, "record_date", ["education_level","enrollment_status","attendance_rate","progress_percent","completion_status"])
health_last = latest_by_res(past_health, "record_date", ["general_health_score","nutrition_score","sleep_quality_score","energy_level_score","bmi"])
df = base.merge(inc_feat, on="resident_id", how="left").merge(vis_feat, on="resident_id", how="left").merge(rec_feat, on="resident_id", how="left").merge(edu_last, on="resident_id", how="left").merge(health_last, on="resident_id", how="left").merge(y, on="resident_id", how="left")
df["incident_next_30d"] = df["incident_next_30d"].fillna(0).astype(int)
count_cols = ["incidents_90d","high_severity_90d","unresolved_90d","visits_90d","safety_concerns_90d","followups_90d","recordings_30d","progress_noted_30d","concerns_flagged_30d","referrals_30d"]
for col in count_cols:
    df[col] = df[col].fillna(0)
health_edu_cols = ["attendance_rate","progress_percent","general_health_score","nutrition_score","sleep_quality_score","energy_level_score","bmi"]
fill_numeric_median(df, health_edu_cols)
cat_cols = ["case_status","case_category","referral_source","reintegration_status","initial_risk_level","current_risk_level","education_level","enrollment_status","completion_status"]
df[cat_cols] = df[cat_cols].fillna("Unknown")
print("Rows:", len(df), "Incident rate:", round(df["incident_next_30d"].mean(), 3))
quick_eda(df, "Resident risk/readiness modeling table", target_col="incident_next_30d", numeric_cols=count_cols + health_edu_cols, categorical_cols=cat_cols)


Rows: 60 Incident rate: 0.233

EDA: Resident risk/readiness modeling table
Shape: (60, 36)

Top missing-value rates:
resident_id             0.0
safehouse_id            0.0
followups_90d           0.0
recordings_30d          0.0
progress_noted_30d      0.0
concerns_flagged_30d    0.0
referrals_30d           0.0
education_level         0.0
enrollment_status       0.0
attendance_rate         0.0

Target distribution / summary: incident_next_30d
count    60.000000
mean      0.233333
std       0.426522
min       0.000000
25%       0.000000
50%       0.000000
75%       0.000000
max       1.000000
Saved EDA plot: /Users/xenorex/RiderProjects/INTEX2/output/ml-predictions/eda-plots/resident_risk_readiness_modeling_table_incident_next_30d_distribution.png

Numeric feature summary:
                        mean     std     min      50%     max
incidents_90d          0.200   0.480   0.000    0.000    2.00
high_severity_90d      0.017   0.129   0.000    0.000    1.00
unresolved_90d         0.033   

## 3) Modeling and Feature Selection

The feature set is selected from fields that would be available at the decision point whenever possible. Predictive models use ensembles or tuned tree-based models when they improve out-of-sample performance. Explanatory models use simpler linear or logistic models when interpretability matters.


In [3]:
target = "incident_next_30d"
num_cols = ["safehouse_id","is_pwd","has_special_needs","family_is_4ps","family_solo_parent","family_indigenous","family_parent_pwd","family_informal_settler","incidents_90d","high_severity_90d","unresolved_90d","visits_90d","safety_concerns_90d","followups_90d","recordings_30d","progress_noted_30d","concerns_flagged_30d","referrals_30d","attendance_rate","progress_percent","general_health_score","nutrition_score","sleep_quality_score","energy_level_score","bmi"]
features = cat_cols + num_cols

X_train, X_test, y_train, y_test = safe_classifier_split(df[features], df[target])

# Explanatory baseline for interpretation.
explanatory = Pipeline([("pre", prep(cat_cols, num_cols)), ("model", LogisticRegression(max_iter=3000))])

incident_candidates = {
    "LogisticRegression": explanatory,
    "RandomForestSmall": Pipeline([("pre", prep(cat_cols, num_cols)), ("model", RandomForestClassifier(n_estimators=200, random_state=42, min_samples_leaf=3))]),
    "GradientBoosting": Pipeline([("pre", prep(cat_cols, num_cols)), ("model", GradientBoostingClassifier(random_state=42))]),
}

# Readiness target.
df["readiness_positive"] = df["reintegration_status"].isin(["Completed", "In Progress"]).astype(int)
X_ready_train, X_ready_test, y_ready_train, y_ready_test = safe_classifier_split(df[features], df["readiness_positive"])

readiness_explanatory = Pipeline([("pre", prep(cat_cols, num_cols)), ("model", LogisticRegression(max_iter=3000))])
readiness_candidates = {
    "LogisticRegression": readiness_explanatory,
    "RandomForestSmall": Pipeline([("pre", prep(cat_cols, num_cols)), ("model", RandomForestClassifier(n_estimators=200, random_state=42, min_samples_leaf=3))]),
    "GradientBoosting": Pipeline([("pre", prep(cat_cols, num_cols)), ("model", GradientBoostingClassifier(random_state=42))]),
}



## 4) Evaluation and Selection

The notebook uses a train/test or time-based holdout split depending on the business problem. Where appropriate, it also uses compact cross-validation, model comparison tables, and lightweight randomized tuning. Metrics are interpreted in business terms rather than treated as abstract statistics.


In [4]:
# Strict holdout discipline: CV on train, single touch on holdout.
X_train_full, X_holdout, y_train_full, y_holdout = safe_classifier_split(df[features], df[target])

report_path = paths.reports_dir / "resident-risk-and-readiness_run_report.json"
prev_report = read_json_if_exists(report_path)

current_profile = dataset_profile(df[features], categorical_cols=cat_cols, numeric_cols=num_cols)
drift = compare_profiles(current_profile, (prev_report or {}).get("data_profile"))

# 1) Incident CV tournament
inc_cv_results = [cv_evaluate_candidate(name, est, X_train_full, y_train_full, task="binary", top_fraction=0.10) for name, est in incident_candidates.items()]
inc_cv_tbl = cv_results_table(inc_cv_results).sort_values(by=["recall_at_top10pct_mean", "pr_auc_mean"], ascending=False)
print("Incident CV tournament:")
print(inc_cv_tbl.to_string(index=False))

inc_selected_cv = select_simplest_within_delta_cv(inc_cv_results, primary_metric="recall_at_top10pct", delta=0.02, higher_is_better=True)
inc_base = inc_selected_cv.estimator
inc_base_name = inc_selected_cv.name

# tune incident
inc_param = {}
if inc_base_name.startswith("RandomForest"):
    inc_param = {"model__n_estimators": [100, 150, 200, 300], "model__max_depth": [None, 3, 5, 8], "model__min_samples_leaf": [1, 3, 5, 8]}
elif inc_base_name.startswith("GradientBoosting"):
    inc_param = {"model__n_estimators": [100, 150, 250], "model__learning_rate": [0.03, 0.05, 0.1], "model__max_depth": [2, 3, 4]}

if inc_param:
    tune_inc = tune_model_randomized(inc_base_name, inc_base, X_train_full, y_train_full, param_distributions=inc_param, task="binary", cv_metric="average_precision", n_iter=10)
    incident_model = tune_inc.best_estimator
    incident_model_name = f"{inc_base_name}+tuned"
    incident_params = tune_inc.best_params
else:
    incident_model = inc_base
    incident_model_name = inc_base_name
    incident_params = {}

# Ablation (incident)
inc_ablation = ablate_feature_groups_one_by_one_cv(
    base_name=incident_model_name,
    estimator=incident_model,
    X=X_train_full,
    y=y_train_full,
    task="binary",
    feature_groups=[[c] for c in features],
    primary_metric="recall_at_top10pct",
    higher_is_better=True,
    top_fraction=0.10,
)
write_ablation_report_json(paths.reports_dir / "resident-risk-and-readiness_incident_ablation_report.json", {"pipeline": "resident-risk-and-readiness", "target": "incident", "table": inc_ablation.to_dict(orient="records")})

# 2) Readiness CV tournament
readiness_y = df["readiness_positive"]
X_ready_full, X_ready_holdout, y_ready_full, y_ready_holdout = safe_classifier_split(df[features], readiness_y)

ready_cv_results = [cv_evaluate_candidate(name, est, X_ready_full, y_ready_full, task="binary", top_fraction=0.10) for name, est in readiness_candidates.items()]
ready_cv_tbl = cv_results_table(ready_cv_results).sort_values(by=["recall_at_top10pct_mean", "pr_auc_mean"], ascending=False)
print("Readiness CV tournament:")
print(ready_cv_tbl.to_string(index=False))

ready_selected_cv = select_simplest_within_delta_cv(ready_cv_results, primary_metric="recall_at_top10pct", delta=0.02, higher_is_better=True)
ready_base = ready_selected_cv.estimator
ready_base_name = ready_selected_cv.name

ready_param = {}
if ready_base_name.startswith("RandomForest"):
    ready_param = {"model__n_estimators": [100, 150, 200, 300], "model__max_depth": [None, 3, 5, 8], "model__min_samples_leaf": [1, 3, 5, 8]}
elif ready_base_name.startswith("GradientBoosting"):
    ready_param = {"model__n_estimators": [100, 150, 250], "model__learning_rate": [0.03, 0.05, 0.1], "model__max_depth": [2, 3, 4]}

if ready_param:
    tune_ready = tune_model_randomized(ready_base_name, ready_base, X_ready_full, y_ready_full, param_distributions=ready_param, task="binary", cv_metric="average_precision", n_iter=10)
    readiness_model = tune_ready.best_estimator
    readiness_model_name = f"{ready_base_name}+tuned"
    readiness_params = tune_ready.best_params
else:
    readiness_model = ready_base
    readiness_model_name = ready_base_name
    readiness_params = {}

ready_ablation = ablate_feature_groups_one_by_one_cv(
    base_name=readiness_model_name,
    estimator=readiness_model,
    X=X_ready_full,
    y=y_ready_full,
    task="binary",
    feature_groups=[[c] for c in features],
    primary_metric="recall_at_top10pct",
    higher_is_better=True,
    top_fraction=0.10,
)
write_ablation_report_json(paths.reports_dir / "resident-risk-and-readiness_readiness_ablation_report.json", {"pipeline": "resident-risk-and-readiness", "target": "readiness", "table": ready_ablation.to_dict(orient="records")})

# 3) Final holdout report (single touch)
incident_model.fit(X_train_full, y_train_full)
readiness_model.fit(X_ready_full, y_ready_full)

from modeling import eval_classification
holdout_proba = incident_model.predict_proba(X_holdout)[:, 1]
holdout_pred = (holdout_proba >= 0.5).astype(int)
incident_holdout = eval_classification(y_holdout, holdout_pred, holdout_proba)

ready_proba = readiness_model.predict_proba(X_ready_holdout)[:, 1]
ready_pred = (ready_proba >= 0.5).astype(int)
readiness_holdout = eval_classification(y_ready_holdout, ready_pred, ready_proba)

allow_export, guardrail = should_export_by_guardrail(
    previous_report=prev_report,
    current_holdout={"pr_auc": float(incident_holdout.get("pr_auc", 0.0))},
    primary_metric="pr_auc",
    min_delta_allowed=0.01,
    higher_is_better=True,
)

write_run_report_json(
    report_path,
    {
        "pipeline": "resident-risk-and-readiness",
        "primary_metric": "recall_at_top10pct",
        "incident": {
            "selected_family": inc_base_name,
            "tuned_name": incident_model_name,
            "tuned_params": incident_params,
            "cv_table": inc_cv_tbl.to_dict(orient="records"),
            "holdout": incident_holdout,
            "horizon_days": incident_horizon_days,
        },
        "readiness": {
            "selected_family": ready_base_name,
            "tuned_name": readiness_model_name,
            "tuned_params": readiness_params,
            "cv_table": ready_cv_tbl.to_dict(orient="records"),
            "holdout": readiness_holdout,
        },
        "guardrail": guardrail,
        "data_profile": current_profile,
        "drift": drift,
    },
)
print("Wrote run report:", report_path)

if not allow_export:
    print("Export blocked by guardrail:", guardrail)

# Keep explanatory reference fits (optional)
explanatory.fit(X_train_full, y_train_full)
readiness_explanatory.fit(X_ready_full, y_ready_full)



Incident CV tournament:
             model  fit_s_mean  predict_s_mean  accuracy_mean  precision_mean  recall_mean  f1_mean  roc_auc_mean  pr_auc_mean  recall_at_top10pct_mean  accuracy_std  precision_std  recall_std   f1_std  roc_auc_std  pr_auc_std  recall_at_top10pct_std
LogisticRegression    0.020809        0.004393       0.777778        0.555556     0.555556 0.555556      0.806397     0.619048                 0.388889      0.076980       0.096225    0.096225 0.096225     0.056823    0.142564                0.240563
 RandomForestSmall    0.095755        0.010145       0.733333        0.000000     0.000000 0.000000      0.733165     0.539683                 0.277778      0.066667       0.000000    0.000000 0.000000     0.035384    0.093043                0.048113
  GradientBoosting    0.026866        0.004289       0.711111        0.166667     0.166667 0.166667      0.602273     0.320833                 0.111111      0.101835       0.288675    0.288675 0.288675     0.126540    0.071

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('pre', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sp

## 5) Causal and Relationship Analysis

The relationship analysis section highlights important features and discusses whether those relationships make sense for the organization. These findings are observational: they can guide hypotheses and strategy, but they are not automatically causal. Any sensitive resident-care decision must remain human-reviewed.


In [5]:
print("Top selected incident-risk features:")
print(top_features(incident_model).to_string(index=False))
print("Top explanatory risk relationships:")
print(top_features(explanatory).to_string(index=False))
print("Top selected readiness features:")
print(top_features(readiness_model).to_string(index=False))
print_business_takeaway("Use resident risk/readiness scores for staff triage only. Sensitive care decisions should always remain human-reviewed.")


Top selected incident-risk features:
                         feature  importance
       num__concerns_flagged_30d    0.897294
              num__followups_90d    0.847550
cat__referral_source_Court Order    0.719025
          num__has_special_needs    0.672068
cat__initial_risk_level_Critical    0.661814
cat__education_level_CollegePrep    0.629401
     cat__current_risk_level_Low    0.559497
              num__referrals_30d    0.549502
    cat__education_level_Primary    0.520061
        num__sleep_quality_score    0.503204
Top explanatory risk relationships:
                         feature  importance
       num__concerns_flagged_30d    0.897294
              num__followups_90d    0.847550
cat__referral_source_Court Order    0.719025
          num__has_special_needs    0.672068
cat__initial_risk_level_Critical    0.661814
cat__education_level_CollegePrep    0.629401
     cat__current_risk_level_Low    0.559497
              num__referrals_30d    0.549502
    cat__education_level_Pr

## 6) Deployment Notes

The final scoring step exports JSON to `output/ml-predictions/`. These files match the API import contract used by `POST /api/ml/import?replace=true` and can be viewed in the deployed staff portal under `/app/ml` or the ML action center.


In [6]:
df_out = df[["resident_id"] + features].copy()
df_out["risk_score"] = incident_model.predict_proba(df_out[features])[:, 1]
df_out["risk_band"] = score_bands(df_out["risk_score"])

# Standardized export name matches the training horizon.
incident_prediction_type = f"resident_incident_{incident_horizon_days}d"

if allow_export:
    export_predictions_json(
        incident_prediction_type,
        "Resident",
        df_out[[
            "resident_id",
            "risk_score",
            "risk_band",
            "safehouse_id",
            "incidents_90d",
            "safety_concerns_90d",
            "followups_90d",
            "recordings_30d",
            "progress_percent",
            "general_health_score",
        ]].assign(horizon_days=incident_horizon_days, export_model=incident_model_name),
        "resident_id",
        "risk_score",
        "risk_band",
    )
else:
    print("Skipping incident export due to guardrail.")


Wrote 60 predictions: /Users/xenorex/RiderProjects/INTEX2/output/ml-predictions/resident_incident_180d.json


In [7]:
ready_out = df[["resident_id"] + features].copy()
ready_out["readiness_score"] = readiness_model.predict_proba(ready_out[features])[:, 1]
ready_out["readiness_band"] = score_bands(ready_out["readiness_score"])
if allow_export:
    export_predictions_json(
        "resident_reintegration_readiness",
        "Resident",
        ready_out[[
            "resident_id",
            "readiness_score",
            "readiness_band",
            "safehouse_id",
            "reintegration_status",
            "progress_percent",
            "general_health_score",
            "incidents_90d",
            "concerns_flagged_30d",
        ]].assign(export_model=readiness_model_name),
        "resident_id",
        "readiness_score",
        "readiness_band",
    )
else:
    print("Skipping readiness export due to guardrail.")



Wrote 60 predictions: /Users/xenorex/RiderProjects/INTEX2/output/ml-predictions/resident_reintegration_readiness.json
